# TiO2 model training and evaluation
Stage C uses the saved model and metrics. It does not retrain during the demonstration. Validation is preliminary: exact input groups are separated, but independent study IDs are unavailable. Outputs are in reported mlogk units, not removal percentages.

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
from IPython.display import Image, display
ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.predict import TiO2Predictor
card = json.loads((ROOT / 'models/model_card.json').read_text())
print('Selected model:', card['model_name'])
print('Training rows:', card['train_rows'], '| held out rows:', card['test_rows'])
print('Numerical prototype enabled:', card['enabled_for_prototype'])
pd.read_csv(ROOT / 'outputs/model_comparison_cv.csv')[['model', 'cv_mae_mean', 'cv_mae_std', 'best_parameters']]

## Untouched final test results
The model was selected using training CV before evaluating these test outcomes. R squared is not percent accuracy. MAE is in the reported transformed target.

In [ ]:
pd.DataFrame({'Selected model': card['test_metrics'], 'Median baseline': card['baseline_test_metrics']}).T

In [ ]:
display(Image(filename=str(ROOT / 'outputs/predicted_vs_measured.png')))

In [ ]:
print('Held out rows passing domain policy:', card['eligible_test_rows'], '/', card['test_rows'])
print('Eligible subset metrics:', card['eligible_test_metrics'])
print('This subset is not a calibrated confidence region.')

## Run one numerical prediction
Use the first eligible held out SMX condition where available, selected by inputs rather than prediction error. This is a published test observation, not a new lab experiment. The default command line example separately uses a training condition and labels it explicitly.

In [ ]:
demo_file = ROOT / 'config/demo_holdout_scenario.json'
if not demo_file.exists(): demo_file = ROOT / 'config/demo_ml_scenario.json'
scenario = json.loads(demo_file.read_text())
predictor = TiO2Predictor()
answer = predictor.predict(scenario)
print('Scenario:', scenario)
print({k: answer[k] for k in ['status', 'prediction', 'prediction_target', 'exact_training_conditions', 'reasons']})
test_predictions = pd.read_csv(ROOT / 'outputs/test_predictions.csv')
test_predictions.loc[test_predictions.source_row.eq(scenario['source_row']), ['Molecule_Name', 'mlogk', 'predicted_mlogk']]

## Check an unsupported request
Methylene blue is absent from this training dataset. The API must return evidence only rather than fabricate a numerical prediction.

In [ ]:
unsupported = dict(scenario, pollutant_id='methylene_blue')
fallback = predictor.predict(unsupported)
print({k: fallback[k] for k in ['status', 'prediction', 'reasons']})

## Limits and next stage
No independent study or lab validation. Phase, size, surface area, wavelength and matrix effects are unmodelled. No prediction interval is claimed and no inverse log conversion is made. Next: integrate the saved model with deterministic evidence screening and recommendation rules for all 15 families.